In [ ]:
"""
# Workshop 2: Data Profiling (EDA)
## ETL Process - Spotify × Grammy Awards

**Objective:** Exploratory Data Analysis of both datasets to understand their structure,
data quality, and define the necessary transformations for the ETL pipeline.
"""

In [ ]:
"""
## 1. Setup and Libraries
"""

In [ ]:
# Import libraries
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
import numpy as np

# Visualization configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

# Configure pandas to display more columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
"""
## 2. Data Loading and Validation
"""

In [ ]:
# Define the base path to the project directory
base_path = Path.cwd().parent

spotify_path = base_path / "data" / "raw" / "spotify_dataset.csv"
grammy_path = base_path / "data" / "raw" / "the_grammy_awards.csv"

# Validate that files exist
print("=" * 60)
print("FILE VALIDATION")
print("=" * 60)
print(f"Spotify dataset exists: {spotify_path.exists()}")
print(f"Grammy dataset exists: {grammy_path.exists()}")

if not spotify_path.exists():
    raise FileNotFoundError(f"Not found: {spotify_path}")
if not grammy_path.exists():
    raise FileNotFoundError(f"Not found: {grammy_path}")

In [ ]:
"""
## 3. Spotify Dataset Analysis
"""

In [ ]:
"""
### 3.1 Structure and Overview
"""

In [ ]:
# Load data
spotify_df = pd.read_csv(spotify_path)

print("=" * 60)
print("SPOTIFY DATASET - GENERAL INFORMATION")
print("=" * 60)
print(f"Dimensions: {spotify_df.shape[0]:,} rows × {spotify_df.shape[1]} columns")
print(f"\nMemory usage: {spotify_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Data types information
print("\n--- Data types ---")
spotify_df.info()

In [ ]:
# First rows
print("\n--- First 5 rows ---")
spotify_df.head()

In [ ]:
# Available columns
print("\n--- Dataset columns ---")
for i, col in enumerate(spotify_df.columns, 1):
    print(f"{i:2d}. {col}")

In [ ]:
"""
### 3.2 Data Quality - Null Values
"""

In [ ]:
# Null values analysis
print("=" * 60)
print("NULL VALUES ANALYSIS - SPOTIFY")
print("=" * 60)

null_counts = spotify_df.isnull().sum()
null_percent = (null_counts / len(spotify_df)) * 100

null_summary = pd.DataFrame({
    'Nulls': null_counts,
    'Percentage': null_percent.round(2)
}).sort_values('Nulls', ascending=False)

print(null_summary)

# Visualization
plt.figure(figsize=(12, 6))
null_summary[null_summary['Nulls'] > 0].plot(kind='bar', y='Percentage', legend=False)
plt.title('Percentage of Null Values by Column - Spotify')
plt.ylabel('% Nulls')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
"""
### 3.3 Duplicates Analysis
"""

In [ ]:
print("=" * 60)
print("DUPLICATES ANALYSIS - SPOTIFY")
print("=" * 60)

# Different duplication levels
duplicates = {
    'Total exact duplicates': spotify_df.duplicated().sum(),
    'By track_id': spotify_df.duplicated(subset=['track_id']).sum(),
    'By track + artist': spotify_df.duplicated(subset=['track_name', 'artists']).sum(),
    'By track + artist + album': spotify_df.duplicated(subset=['track_name', 'artists', 'album_name']).sum(),
    'By track + artist + album + genre': spotify_df.duplicated(subset=['track_name', 'artists', 'album_name', 'track_genre']).sum()
}

for criteria, count in duplicates.items():
    print(f"{criteria:40s}: {count:>6,}")

# Analysis of duplicates with same track_id but different genre
track_id_counts = spotify_df['track_id'].value_counts()
multi_genre_tracks = track_id_counts[track_id_counts > 1]

print(f"\nTracks with multiple genres: {len(multi_genre_tracks):,}")
print(f"Example of track with multiple genres:")
example_track = multi_genre_tracks.index[0]
spotify_df[spotify_df['track_id'] == example_track][['track_name', 'artists', 'track_genre']].head()

In [ ]:
"""
### 3.4 Descriptive Statistics
"""

In [ ]:
print("=" * 60)
print("DESCRIPTIVE STATISTICS - SPOTIFY")
print("=" * 60)

# Numeric variables
numeric_cols = spotify_df.select_dtypes(include=[np.number]).columns
print(f"\nNumeric variables: {len(numeric_cols)}")
spotify_df[numeric_cols].describe().round(3)

In [ ]:
"""
### 3.5 Audio Features Analysis
"""

In [ ]:
# Main audio features
audio_features = ['danceability', 'energy', 'loudness', 'speechiness', 
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

print("=" * 60)
print("AUDIO FEATURES ANALYSIS")
print("=" * 60)

# Correlation matrix
plt.figure(figsize=(12, 10))
corr_matrix = spotify_df[audio_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, fmt='.2f', cbar_kws={"shrink": .8})
plt.title('Audio Features Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for idx, col in enumerate(audio_features):
    sns.histplot(spotify_df[col], ax=axes[idx], kde=True, bins=30)
    axes[idx].set_title(f'Distribution of {col}')
    axes[idx].set_xlabel('')

# Remove empty subplot if exists
if len(audio_features) < len(axes):
    fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

In [ ]:
"""
### 3.6 Popularity Analysis
"""

In [ ]:
print("=" * 60)
print("POPULARITY ANALYSIS")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
sns.histplot(spotify_df['popularity'], bins=30, ax=axes[0], kde=True)
axes[0].set_title('Popularity Distribution (Total)')
axes[0].set_xlabel('Popularity')
axes[0].axvline(spotify_df['popularity'].mean(), color='red', linestyle='--', 
                label=f'Mean: {spotify_df["popularity"].mean():.1f}')
axes[0].legend()

# Excluding zeros
popularity_nonzero = spotify_df[spotify_df['popularity'] > 0]['popularity']
sns.histplot(popularity_nonzero, bins=30, ax=axes[1], kde=True, color='green')
axes[1].set_title('Popularity Distribution (Excluding 0)')
axes[1].set_xlabel('Popularity')
axes[1].axvline(popularity_nonzero.mean(), color='red', linestyle='--',
                label=f'Mean: {popularity_nonzero.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nTracks with popularity = 0: {(spotify_df['popularity'] == 0).sum():,} ({(spotify_df['popularity'] == 0).mean()*100:.1f}%)")

In [ ]:
"""
### 3.7 Artists and Genres Analysis
"""

In [ ]:
print("=" * 60)
print("TOP ARTISTS AND GENRES")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top 10 artists by number of tracks
top_artists = spotify_df['artists'].value_counts().head(10)
sns.barplot(x=top_artists.values, y=top_artists.index, ax=axes[0,0])
axes[0,0].set_title('Top 10 Artists by Number of Tracks')
axes[0,0].set_xlabel('Number of Tracks')

# Top 10 genres
top_genres = spotify_df['track_genre'].value_counts().head(10)
sns.barplot(x=top_genres.values, y=top_genres.index, ax=axes[0,1])
axes[0,1].set_title('Top 10 Genres')
axes[0,1].set_xlabel('Count')

# Top artists by average popularity (min 3 tracks)
artist_popularity = spotify_df.groupby('artists').agg({
    'popularity': 'mean',
    'track_id': 'count'
}).reset_index()
artist_popularity.columns = ['artists', 'avg_popularity', 'track_count']
artist_popularity = artist_popularity[artist_popularity['track_count'] >= 3]
top_popular_artists = artist_popularity.nlargest(10, 'avg_popularity')

sns.barplot(x='avg_popularity', y='artists', data=top_popular_artists, ax=axes[1,0])
axes[1,0].set_title('Top Artists by Average Popularity (min 3 tracks)')
axes[1,0].set_xlabel('Average Popularity')

# Popularity by genre (top 10 genres)
top_genres_list = top_genres.index
filtered_df = spotify_df[spotify_df['track_genre'].isin(top_genres_list)]
sns.boxplot(data=filtered_df, x='track_genre', y='popularity', ax=axes[1,1])
axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].set_title('Popularity Distribution by Genre')

plt.tight_layout()
plt.show()

# Statistics
print(f"\nTotal unique artists: {spotify_df['artists'].nunique():,}")
print(f"Total unique genres: {spotify_df['track_genre'].nunique():,}")
print(f"Total unique tracks (by ID): {spotify_df['track_id'].nunique():,}")

In [ ]:
"""
## 4. Grammy Dataset Analysis
"""

In [ ]:
"""
### 4.1 Structure and Overview
"""

In [ ]:
grammy_df = pd.read_csv(grammy_path)

print("=" * 60)
print("GRAMMY DATASET - GENERAL INFORMATION")
print("=" * 60)
print(f"Dimensions: {grammy_df.shape[0]:,} rows × {grammy_df.shape[1]} columns")

print("\n--- Data types ---")
grammy_df.info()

In [ ]:
print("\n--- First 5 rows ---")
grammy_df.head()

In [ ]:
"""
### 4.2 Data Quality
"""

In [ ]:
print("=" * 60)
print("NULL VALUES ANALYSIS - GRAMMY")
print("=" * 60)

null_counts_g = grammy_df.isnull().sum()
null_percent_g = (null_counts_g / len(grammy_df)) * 100

null_summary_g = pd.DataFrame({
    'Nulls': null_counts_g,
    'Percentage': null_percent_g.round(2)
}).sort_values('Nulls', ascending=False)

print(null_summary_g)

# Visualization
plt.figure(figsize=(10, 6))
null_summary_g[null_summary_g['Nulls'] > 0].plot(kind='bar', y='Percentage', legend=False, color='coral')
plt.title('Percentage of Null Values by Column - Grammy')
plt.ylabel('% Nulls')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
"""
### 4.3 Duplicates Analysis
"""

In [ ]:
print("=" * 60)
print("DUPLICATES ANALYSIS - GRAMMY")
print("=" * 60)

print(f"Exact duplicate rows: {grammy_df.duplicated().sum():,}")

# Duplicates by year, category, nominee and artist
dup_criteria = ['year', 'category', 'nominee', 'artist']
print(f"Duplicates by {dup_criteria}: {grammy_df.duplicated(subset=dup_criteria).sum():,}")

In [ ]:
"""
### 4.4 Temporal Analysis
"""

In [ ]:
print("=" * 60)
print("TEMPORAL ANALYSIS - GRAMMY")
print("=" * 60)

# Convert year to numeric
grammy_df['year'] = pd.to_numeric(grammy_df['year'], errors='coerce')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Records per year
yearly_counts = grammy_df['year'].value_counts().sort_index()
yearly_counts.plot(kind='line', ax=axes[0,0], marker='o', markersize=3)
axes[0,0].set_title('Number of Grammy Records per Year')
axes[0,0].set_xlabel('Year')
axes[0,0].set_ylabel('Count')

# Distribution by decade
grammy_df['decade'] = (grammy_df['year'] // 10) * 10
decade_counts = grammy_df['decade'].value_counts().sort_index()
decade_counts.plot(kind='bar', ax=axes[0,1], color='skyblue')
axes[0,1].set_title('Records by Decade')
axes[0,1].set_xlabel('Decade')

# Top categories
top_categories = grammy_df['category'].value_counts().head(10)
sns.barplot(x=top_categories.values, y=top_categories.index, ax=axes[1,0])
axes[1,0].set_title('Top 10 Grammy Categories')

# Remove empty subplot
fig.delaxes(axes[1,1])

plt.tight_layout()
plt.show()

print(f"\nYear range: {grammy_df['year'].min():.0f} - {grammy_df['year'].max():.0f}")
print(f"Year with most records: {yearly_counts.idxmax():.0f} ({yearly_counts.max()} records)")
print(f"\nNote: All {len(grammy_df):,} records in this dataset are Grammy winners")

In [ ]:
"""
### 4.5 Artists Analysis
"""

In [ ]:
print("=" * 60)
print("ARTISTS ANALYSIS - GRAMMY")
print("=" * 60)

# Clean null artists
grammy_artists_clean = grammy_df[grammy_df['artist'].notna() & (grammy_df['artist'] != '')]

# Top artists by nominations (all records are winners in this dataset)
artist_nominations = grammy_artists_clean['artist'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top artists by nominations
sns.barplot(x=artist_nominations.values, y=artist_nominations.index, ax=axes[0])
axes[0].set_title('Top 10 Artists by Grammy Wins')

# Top artists by decade (2010s vs 2020s)
recent_artists_2010s = grammy_df[(grammy_df['year'] >= 2010) & (grammy_df['year'] < 2020)]['artist'].value_counts().head(8)
recent_artists_2020s = grammy_df[grammy_df['year'] >= 2020]['artist'].value_counts().head(8)

# Most data decade
if len(recent_artists_2020s) > 3:
    sns.barplot(x=recent_artists_2020s.values, y=recent_artists_2020s.index, ax=axes[1])
    axes[1].set_title('Top Artists (2020s)')
else:
    sns.barplot(x=recent_artists_2010s.values, y=recent_artists_2010s.index, ax=axes[1])
    axes[1].set_title('Top Artists (2010s)')

plt.tight_layout()
plt.show()

# Artist statistics
print(f"\nUnique artists: {grammy_artists_clean['artist'].nunique():,}")
print(f"Total Grammy winners in dataset: {len(grammy_df):,}")

In [ ]:
"""
## 5. Comparative Pre-Merge Analysis
"""

In [ ]:
"""
### 5.1 Artist Name Normalization
"""

In [ ]:
print("=" * 60)
print("COMPARATIVE ANALYSIS: ARTIST COVERAGE")
print("=" * 60)

# Simulate the normalization that ETL will do
def normalize_artist_spotify(artist):
    """Simulates transform_spotify.py normalization"""
    if pd.isna(artist):
        return 'unknown'
    return str(artist).lower().strip().split(';')[0].split(',')[0].strip()

def normalize_artist_grammy(artist):
    """Simulates transform_grammys.py normalization"""
    if pd.isna(artist) or artist == '':
        return 'unknown'
    return str(artist).lower().strip()

# Apply normalization
spotify_artists_norm = set(spotify_df['artists'].apply(normalize_artist_spotify).unique())
grammy_artists_norm = set(grammy_df['artist'].apply(normalize_artist_grammy).unique())

# Coverage analysis
common_artists = spotify_artists_norm.intersection(grammy_artists_norm)
only_spotify = spotify_artists_norm - grammy_artists_norm
only_grammy = grammy_artists_norm - spotify_artists_norm

print(f"Unique artists in Spotify (normalized): {len(spotify_artists_norm):,}")
print(f"Unique artists in Grammy (normalized): {len(grammy_artists_norm):,}")
print(f"Artists in common: {len(common_artists):,}")
print(f"Only in Spotify: {len(only_spotify):,}")
print(f"Only in Grammy: {len(only_grammy):,}")

coverage_pct = len(common_artists) / len(spotify_artists_norm) * 100
print(f"\n📊 Expected merge coverage: {coverage_pct:.2f}% of Spotify artists have Grammy data")

In [ ]:
# Coverage visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Simplified Venn diagram (bars)
categories = ['Only Spotify', 'In common', 'Only Grammy']
values = [len(only_spotify), len(common_artists), len(only_grammy)]
colors = ['#ff9999', '#66b3ff', '#99ff99']

axes[0].bar(categories, values, color=colors)
axes[0].set_title('Artist Coverage Across Datasets')
axes[0].set_ylabel('Number of Artists')
for i, v in enumerate(values):
    axes[0].text(i, v + max(values)*0.01, f'{v:,}', ha='center', fontweight='bold')

# Match examples
sample_matches = list(common_artists)[:10]

axes[1].axis('off')
table_data = [
    ['Metric', 'Value'],
    ['Total Spotify', f'{len(spotify_artists_norm):,}'],
    ['Total Grammy', f'{len(grammy_artists_norm):,}'],
    ['Exact match', f'{len(common_artists):,}'],
    ['Coverage %', f'{coverage_pct:.2f}%'],
    ['', ''],
    ['Match examples:', ''],
] + [[f'  {i+1}.', artist] for i, artist in enumerate(sample_matches[:5])]

table = axes[1].table(cellText=table_data, loc='center', cellLoc='left',
                      colWidths=[0.4, 0.6])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)
axes[1].set_title('Coverage Summary', pad=20)

plt.tight_layout()
plt.show()

In [ ]:
"""
### 5.2 Merge Case Examples
"""

In [ ]:
print("\n--- Examples of artists that will MATCH ---")
for artist in list(common_artists)[:5]:
    spotify_count = spotify_df[spotify_df['artists'].str.lower().str.contains(artist, na=False)].shape[0]
    grammy_count = grammy_df[grammy_df['artist'].str.lower().str.contains(artist, na=False)].shape[0]
    print(f"  '{artist}': {spotify_count} tracks in Spotify, {grammy_count} Grammy records")

print("\n--- Examples of artists in Spotify WITHOUT match ---")
for artist in list(only_spotify)[:5]:
    print(f"  '{artist}'")

In [ ]:
"""
## 6. Key Findings and Transformation Decisions
"""

In [ ]:
print("=" * 70)
print("EXECUTIVE SUMMARY AND TRANSFORMATION DECISIONS")
print("=" * 70)

# Calculate key metrics
spotify_tracks_with_0_pop = (spotify_df['popularity'] == 0).sum()
spotify_duplicates_exact = spotify_df.duplicated(subset=['track_name', 'artists', 'album_name', 'track_genre']).sum()

eda_summary = {
    'spotify_total_rows': len(spotify_df),
    'spotify_unique_tracks': spotify_df['track_id'].nunique(),
    'spotify_unique_artists': spotify_df['artists'].nunique(),
    'spotify_unique_genres': spotify_df['track_genre'].nunique(),
    'spotify_nulls_critical': spotify_df[['artists', 'album_name', 'track_name']].isnull().sum().sum(),
    'spotify_tracks_popularity_0': spotify_tracks_with_0_pop,
    'spotify_duplicates_exact': spotify_duplicates_exact,
    
    'grammy_total_rows': len(grammy_df),
    'grammy_year_range': f"{grammy_df['year'].min():.0f}-{grammy_df['year'].max():.0f}",
    'grammy_unique_artists': grammy_df['artist'].nunique(),
    'grammy_unique_categories': grammy_df['category'].nunique(),
    'grammy_total_winners': (grammy_df['winner'] == True).sum(),
    
    'merge_expected_coverage_pct': round(coverage_pct, 2),
    'merge_common_artists': len(common_artists),
}

print("\n--- KEY METRICS ---")
for key, value in eda_summary.items():
    print(f"{key:35s}: {value}")

In [ ]:
"""
## 7. Transformation Decisions for the ETL Pipeline
"""

In [ ]:
from IPython.display import Markdown

decisions_md = f"""
| # | Finding | Decision in Pipeline |
|---|---------|----------------------|
| 1 | **Duplicates in Spotify**: {eda_summary['spotify_duplicates_exact']:,} duplicate rows by (track, artist, album, genre) | Remove exact duplicates in `transform_spotify.py` |
| 2 | **Artists with separators**: Multiple artists separated by `;` | Take only first artist as `artist_norm` for join |
| 3 | **Popularity = 0**: {eda_summary['spotify_tracks_popularity_0']:,} tracks ({eda_summary['spotify_tracks_popularity_0']/eda_summary['spotify_total_rows']*100:.1f}%) | Keep them (valid tracks, not errors) |
| 4 | **Multi-genre**: One track can have multiple genres | Preserve in fact_track grain (track_id × genre_key) |
| 5 | **Merge coverage**: ~{eda_summary['merge_expected_coverage_pct']:.1f}% of artists | Use LEFT JOIN to not lose Spotify tracks |
| 6 | **Null artists in Grammy**: {grammy_df['artist'].isnull().sum():,} records | Map to "unknown" to avoid losing nominations |
| 7 | **Data types**: Years as string, winner as string | Cast to numeric/boolean in transformation |
| 8 | **Critical nulls**: {eda_summary['spotify_nulls_critical']} in Spotify key cols | Drop rows with nulls in (artists, album_name, track_name) |
"""

display(Markdown(decisions_md))

In [ ]:
print("\n" + "=" * 70)
print("EDA COMPLETED - Data ready for ETL pipeline")
print("=" * 70)

# Save summary for reference
import json
summary_path = base_path / "data" / "processed" / "eda_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)

with open(summary_path, 'w') as f:
    json.dump(eda_summary, f, indent=2, default=str)

print(f"\nSummary saved to: {summary_path}")